# Streaming Tool Use with Claude

This cookbook demonstrates how to use Claude's streaming API with tool use. Streaming lets you process responses incrementally as they arrive, while tool use gives Claude the ability to call external functions. Combining both is essential for responsive agentic workflows.

## What you'll learn

- How to stream responses that include tool calls
- How to handle the different event types in a streaming tool use response
- How to collect tool inputs incrementally as they stream in
- How to build a complete agentic loop with streaming

## Prerequisites

- An Anthropic API key (set as `ANTHROPIC_API_KEY` environment variable)
- The `anthropic` Python SDK

## Setup

In [ ]:
%pip install anthropic

In [ ]:
import json

import anthropic

client = anthropic.Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

## 1. Understanding Streaming Events

When you stream a response from Claude that includes tool use, you'll receive a sequence of Server-Sent Events (SSEs). The key events for tool use are:

| Event | Description |
|-------|-------------|
| `message_start` | Contains initial message metadata |
| `content_block_start` | Starts a new content block (text or tool_use) |
| `content_block_delta` | Incremental update — `text_delta` for text, `input_json_delta` for tool inputs |
| `content_block_stop` | Marks the end of a content block |
| `message_delta` | Contains the final `stop_reason` (`end_turn` or `tool_use`) |
| `message_stop` | Final event |

When `stop_reason` is `tool_use`, Claude wants to call a tool. You need to execute it and return the result.

## 2. Define Tools

We'll use a simple weather tool to illustrate streaming tool use. In a real application, you'd replace the mock implementation with actual API calls.

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a given location. Returns temperature in Celsius and a brief description.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and country, e.g. 'London, UK' or 'Tokyo, Japan'",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit. Defaults to celsius.",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "get_forecast",
        "description": "Get a 3-day weather forecast for a given location.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and country, e.g. 'Paris, France'",
                },
            },
            "required": ["location"],
        },
    },
]


def get_weather(location: str, unit: str = "celsius") -> dict:
    """Mock weather API. Replace with a real API call in production."""
    mock_data = {
        "london, uk": {"celsius": 12, "fahrenheit": 54, "description": "Cloudy with light rain"},
        "tokyo, japan": {"celsius": 22, "fahrenheit": 72, "description": "Partly cloudy"},
        "sydney, australia": {"celsius": 28, "fahrenheit": 82, "description": "Sunny"},
        "new york, us": {"celsius": 8, "fahrenheit": 46, "description": "Clear skies"},
        "paris, france": {"celsius": 15, "fahrenheit": 59, "description": "Overcast"},
    }
    data = mock_data.get(location.lower(), {"celsius": 20, "fahrenheit": 68, "description": "Mild"})
    temp = data[unit] if unit in data else data["celsius"]
    return {"location": location, "temperature": temp, "unit": unit, "description": data["description"]}


def get_forecast(location: str) -> dict:
    """Mock forecast API. Replace with a real API call in production."""
    return {
        "location": location,
        "forecast": [
            {"day": "Today", "high": 15, "low": 8, "description": "Partly cloudy"},
            {"day": "Tomorrow", "high": 18, "low": 10, "description": "Sunny"},
            {"day": "Day after", "high": 12, "low": 6, "description": "Rainy"},
        ],
    }


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch tool calls to the appropriate function."""
    if tool_name == "get_weather":
        result = get_weather(**tool_input)
    elif tool_name == "get_forecast":
        result = get_forecast(**tool_input)
    else:
        result = {"error": f"Unknown tool: {tool_name}"}
    return json.dumps(result)

## 3. Low-Level Streaming: Handling Events Manually

The most transparent way to stream with tool use is to handle each event directly. This gives you full control over what happens as the response arrives.

The key challenge with streaming tool use is that tool inputs arrive as **incremental JSON fragments** via `input_json_delta` events. You need to accumulate these fragments and parse the complete JSON once the block is done.

In [ ]:
def stream_with_tools_low_level(messages: list) -> tuple[list, str]:
    """
    Stream a response from Claude, handling tool use at the raw event level.

    Returns:
        content_blocks: The complete list of content blocks from the response
        stop_reason: Why the stream ended ('end_turn' or 'tool_use')
    """
    content_blocks = []
    current_block = None
    current_tool_input_json = ""
    stop_reason = None

    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=1024,
        tools=tools,
        messages=messages,
    ) as stream:
        for event in stream:
            event_type = event.type

            if event_type == "content_block_start":
                block = event.content_block
                if block.type == "text":
                    current_block = {"type": "text", "text": ""}
                    current_tool_input_json = ""
                elif block.type == "tool_use":
                    current_block = {"type": "tool_use", "id": block.id, "name": block.name, "input": {}}
                    current_tool_input_json = ""
                    print(f"\n[Tool call started: {block.name}]", end="", flush=True)

            elif event_type == "content_block_delta":
                delta = event.delta
                if delta.type == "text_delta":
                    # Stream text directly to the user
                    print(delta.text, end="", flush=True)
                    if current_block and current_block["type"] == "text":
                        current_block["text"] += delta.text
                elif delta.type == "input_json_delta":
                    # Accumulate JSON fragments for tool input
                    current_tool_input_json += delta.partial_json

            elif event_type == "content_block_stop":
                if current_block:
                    if current_block["type"] == "tool_use" and current_tool_input_json:
                        # Parse the accumulated JSON once the block is complete
                        current_block["input"] = json.loads(current_tool_input_json)
                        print(f" → input: {current_block['input']}]", flush=True)
                    content_blocks.append(current_block)
                    current_block = None
                    current_tool_input_json = ""

            elif event_type == "message_delta":
                stop_reason = event.delta.stop_reason

    return content_blocks, stop_reason


# Test with a simple query that requires a tool call
print("=" * 60)
print("Asking about the weather in London...")
print("=" * 60)

messages = [{"role": "user", "content": "What's the weather like in London, UK right now?"}]
content_blocks, stop_reason = stream_with_tools_low_level(messages)

print(f"\n\nStop reason: {stop_reason}")
print(f"Content blocks received: {len(content_blocks)}")

## 4. Complete Agentic Loop with Streaming

A single turn isn't enough for a real agent — Claude may need to call multiple tools before giving a final answer. Here we build a complete agentic loop:

1. Send the user's message
2. Stream Claude's response
3. If `stop_reason == 'tool_use'`, execute the tools and add results to the conversation
4. Continue streaming until `stop_reason == 'end_turn'`

In [ ]:
def run_agentic_loop(user_message: str, verbose: bool = True) -> str:
    """
    Run a complete streaming agentic loop: stream → tool call → stream → final answer.

    Returns the final text response from Claude.
    """
    messages = [{"role": "user", "content": user_message}]
    final_response = ""
    turn = 0

    if verbose:
        print(f"User: {user_message}\n")
        print("-" * 60)

    while True:
        turn += 1
        if verbose:
            print(f"[Turn {turn}] Claude: ", end="", flush=True)

        content_blocks, stop_reason = stream_with_tools_low_level(messages)

        # Add Claude's response to the conversation history
        messages.append({"role": "assistant", "content": content_blocks})

        if stop_reason == "end_turn":
            # Extract the final text response
            for block in content_blocks:
                if block["type"] == "text":
                    final_response = block["text"]
            break

        if stop_reason == "tool_use":
            # Execute all requested tool calls
            tool_results = []
            for block in content_blocks:
                if block["type"] != "tool_use":
                    continue

                tool_name = block["name"]
                tool_input = block["input"]
                tool_use_id = block["id"]

                if verbose:
                    print(f"\n[Executing {tool_name}({tool_input})]")

                result = execute_tool(tool_name, tool_input)

                if verbose:
                    print(f"[Result: {result}]")

                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_id,
                        "content": result,
                    }
                )

            # Add tool results to the conversation and continue
            messages.append({"role": "user", "content": tool_results})

            if verbose:
                print(f"\n[Turn {turn + 1}] Claude: ", end="", flush=True)

        else:
            # Unexpected stop reason
            break

    return final_response


# Example: Single tool call
print("=" * 60)
print("Example 1: Single tool call")
print("=" * 60)
run_agentic_loop("What's the current temperature in Tokyo, Japan? Give me the answer in Fahrenheit.")

In [ ]:
# Example: Multiple tool calls in sequence
print("=" * 60)
print("Example 2: Multiple tools called in sequence")
print("=" * 60)
run_agentic_loop(
    "I'm planning a trip. Can you check the current weather AND the 3-day forecast for Paris, France?"
)

## 5. High-Level Streaming with the `stream()` Helper

The Anthropic SDK also provides higher-level helpers that simplify streaming. The `stream()` context manager gives you convenience methods like `text_stream` for easy text output. However, for tool use, you still need to handle events manually — the low-level approach above is the most reliable.

Here's a cleaner version using `get_final_message()` to retrieve the complete response after streaming, which is useful when you want the full message object:

In [ ]:
def stream_and_get_tool_calls(messages: list) -> anthropic.types.Message:
    """
    Stream a response and return the complete message object.
    Prints text as it streams, then returns the full message for tool processing.
    """
    with client.messages.stream(
        model=MODEL_NAME,
        max_tokens=1024,
        tools=tools,
        messages=messages,
    ) as stream:
        # Stream text output in real-time
        for text_chunk in stream.text_stream:
            print(text_chunk, end="", flush=True)

        # Get the complete message with all content blocks (including tool_use)
        return stream.get_final_message()


def run_agentic_loop_highlevel(user_message: str) -> str:
    """
    Agentic loop using the high-level stream helper + get_final_message().
    """
    messages = [{"role": "user", "content": user_message}]
    print(f"User: {user_message}\n")
    print("-" * 60)

    while True:
        print("Claude: ", end="", flush=True)
        response = stream_and_get_tool_calls(messages)
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            # Extract final text
            return next((b.text for b in response.content if b.type == "text"), "")

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue
                print(f"\n[Tool: {block.name}({block.input})]")
                result = execute_tool(block.name, block.input)
                print(f"[Result: {result}]\n")
                tool_results.append(
                    {"type": "tool_result", "tool_use_id": block.id, "content": result}
                )
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return ""


print("=" * 60)
print("High-level streaming with get_final_message()")
print("=" * 60)
run_agentic_loop_highlevel("What's the weather in Sydney, Australia?")

## 6. Streaming Progress Updates During Long Tool Calls

One powerful pattern is having Claude stream a "thinking out loud" text block before making a tool call. This improves perceived responsiveness — users see activity while the tool is executing.

Claude naturally does this when given an appropriate system prompt:

In [ ]:
def run_with_progress_narration(user_message: str) -> None:
    """
    Demonstrate Claude narrating its steps as it streams, then calling tools.
    """
    system = (
        "You are a helpful weather assistant. Before calling a tool, briefly tell the user "
        "what you're about to look up. Keep your narration concise (one sentence)."
    )
    messages = [{"role": "user", "content": user_message}]

    print(f"User: {user_message}\n")
    print("-" * 60)

    while True:
        print("Claude: ", end="", flush=True)
        content_blocks = []
        current_block = None
        current_json = ""
        stop_reason = None

        with client.messages.stream(
            model=MODEL_NAME,
            max_tokens=1024,
            system=system,
            tools=tools,
            messages=messages,
        ) as stream:
            for event in stream:
                if event.type == "content_block_start":
                    block = event.content_block
                    if block.type == "text":
                        current_block = {"type": "text", "text": ""}
                    elif block.type == "tool_use":
                        current_block = {"type": "tool_use", "id": block.id, "name": block.name, "input": {}}
                        current_json = ""
                        print(f"\n  → Calling {block.name}...", end="", flush=True)

                elif event.type == "content_block_delta":
                    if event.delta.type == "text_delta":
                        print(event.delta.text, end="", flush=True)
                        if current_block and current_block["type"] == "text":
                            current_block["text"] += event.delta.text
                    elif event.delta.type == "input_json_delta":
                        current_json += event.delta.partial_json

                elif event.type == "content_block_stop":
                    if current_block:
                        if current_block["type"] == "tool_use" and current_json:
                            current_block["input"] = json.loads(current_json)
                        content_blocks.append(current_block)
                        current_block = None

                elif event.type == "message_delta":
                    stop_reason = event.delta.stop_reason

        messages.append({"role": "assistant", "content": content_blocks})

        if stop_reason == "end_turn":
            print()  # final newline
            break

        if stop_reason == "tool_use":
            tool_results = []
            for block in content_blocks:
                if block["type"] == "tool_use":
                    result = execute_tool(block["name"], block["input"])
                    print(f" done.", flush=True)
                    tool_results.append(
                        {"type": "tool_result", "tool_use_id": block["id"], "content": result}
                    )
            messages.append({"role": "user", "content": tool_results})
        else:
            break


run_with_progress_narration(
    "I'm deciding between traveling to London or New York next week. "
    "Can you check the weather in both cities and give me a recommendation?"
)

## Summary

Here's what we covered:

**Key patterns for streaming tool use:**

1. **Accumulate `input_json_delta` fragments** into a string buffer, then `json.loads()` when the block ends — never try to parse partial JSON.

2. **Check `stop_reason`** after the stream ends:
   - `"end_turn"` → Claude is done, extract the text response
   - `"tool_use"` → execute the tools and continue the loop

3. **Build the conversation history** correctly:
   - After each turn: `messages.append({"role": "assistant", "content": content_blocks})`
   - After tool execution: `messages.append({"role": "user", "content": tool_results})`

4. **Use `get_final_message()`** when you want a clean Message object instead of managing blocks manually — but you'll still need to check `stop_reason` and handle tool results yourself.

**Further reading:**
- [Tool use documentation](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Streaming documentation](https://docs.anthropic.com/en/docs/build-with-claude/streaming)
- [Python SDK reference](https://github.com/anthropics/anthropic-sdk-python)